# 02 — EDA
Exploratory analysis of the combined panel: shape, target distribution, missingness, top correlations, feature groups, and an event-study preview around news days.

In [3]:
# Kernel bootstrap — run first. Finds project root whether Jupyter was launched
# from notebooks/ or from the repo root; installs any missing packages.
import sys, os
def _find_root():
    """Walk up from cwd looking for the folder that contains src/ + requirements.txt."""
    for base in [os.getcwd(), os.path.abspath(".."), os.path.abspath(".")]:
        for cand in [base, os.path.dirname(base)]:
            if os.path.isfile(os.path.join(cand, "requirements.txt")) and os.path.isdir(os.path.join(cand, "src")):
                return cand
    return os.path.abspath("..")
_project_root = _find_root()
if _project_root not in sys.path: sys.path.insert(0, _project_root)
os.chdir(_project_root)
from src.bootstrap import ensure
ensure({'pandas_datareader':'pandas-datareader','ta':'ta','yaml':'pyyaml',
        'yfinance':'yfinance','xgboost':'xgboost','optuna':'optuna',
        'dotenv':'python-dotenv'})
print("bootstrap OK. project_root =", _project_root)


bootstrap OK. project_root = /home/yazadanparast/Downloads/eu-midcap-echo


In [4]:
%load_ext autoreload
%autoreload 2
import pandas as pd, numpy as np, os
from pathlib import Path
parquet = Path('data/processed/df_combined.parquet')
if not parquet.exists():
    print('df_combined.parquet not found — building features now (this may take a minute)...')
    # Try to download raw data first if missing
    if not Path('data/raw/yf_all.parquet').exists():
        from src.data.download_yf import download as dl_yf
        dl_yf()
    if not Path('data/raw/macro_fred.parquet').exists():
        from src.data.download_fred import download as dl_fred
        dl_fred()
    if not Path('data/raw/news_headlines.parquet').exists():
        from src.data.download_dgap import download as dl_news
        try: dl_news()
        except Exception as e: print(f'news download skipped: {e}')
    from src.features.build_features import build_dataset
    df, _ = build_dataset()
else:
    df = pd.read_parquet(parquet)
print('shape:', df.shape)
df.head()


shape: (994718, 178)


,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Ticker,...,sector_retail,sector_semis,sector_shipping,sector_steel,sector_tech,sector_telecom,sector_utilities,disclosure_polarity_neg,disclosure_polarity_none,disclosure_polarity_pos
0,2026-10-19,59.099998,59.700001,59.099998,59.700001,59.700001,4048,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
1,2026-10-20,60.299999,60.299999,60.000000,60.099998,60.099998,2739,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
2,2026-10-21,60.200001,60.900002,60.200001,60.900002,60.900002,2102,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
3,2026-10-22,60.200001,61.200001,60.099998,60.299999,60.299999,8438,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
4,2026-10-23,60.299999,60.500000,60.099998,60.200001,60.200001,2508,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0


In [5]:
# Target balance
print('target rate (P(up in 20d)):', round(df.is_positive_growth_20d_future.mean(),3))
df['is_positive_growth_20d_future'].value_counts(dropna=False)

target rate (P(up in 20d)): 0.546


is_positive_growth_20d_future
1    542753
0    451965
Name: count, dtype: int64

In [6]:
# Tickers, date range
print('tickers:', df['Ticker'].nunique())
print('dates:', df['Date'].min(), '→', df['Date'].max())
print('rows per ticker (median):', int(df.groupby('Ticker').size().median()))

tickers: 196
dates: 2005-01-03 00:00:00 → 2026-11-20 00:00:00
rows per ticker (median): 5567


In [7]:
# Feature groups
growth = [c for c in df.columns if c.startswith('growth_') and 'future' not in c]
ta = [c for c in df.columns if c.startswith(('momentum_','trend_','volatility_','volume_','others_'))]
sent = ['sentiment_score','events_pos_5d','events_neg_5d','events_all_10d','disclosure_today']
macro = [c for c in ['us_10y','us_5y','us_1y','fed_funds','ted_spread','vix','usd_eur'] if c in df.columns]
print(f'growth lags: {len(growth)} | ta-lib indicators: {len(ta)} | sentiment: {len([c for c in sent if c in df.columns])} | macro: {len(macro)}')
print('total columns:', len(df.columns))

growth lags: 12 | ta-lib indicators: 69 | sentiment: 5 | macro: 7
total columns: 178


In [8]:
# Top correlations with 20d-forward target
num = df.select_dtypes('number').drop(columns=[c for c in ['is_positive_growth_20d_future','is_positive_growth_5d_future'] if c in df.columns], errors='ignore')
target = df['is_positive_growth_20d_future']
corrs = num.corrwith(target).dropna().sort_values(key=lambda x: x.abs(), ascending=False)
corrs.head(20)

/home/yazadanparast/.local/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/yazadanparast/.local/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Month_May              -0.048608
momentum_pvo_signal     0.045676
Month_December          0.045579
Month_March             0.042502
ln_volume               0.041461
momentum_pvo            0.038792
trend_mass_index        0.037532
ted_spread             -0.027350
usd_eur                -0.026339
events_all_10d         -0.026302
us_10y                 -0.025845
days_since_pos_event    0.024225
vix                     0.023462
momentum_tsi           -0.022692
momentum_rsi           -0.022573
us_5y                  -0.022438
days_since_neg_event    0.020640
Month_September        -0.018460
events_pos_5d          -0.017985
trend_vortex_ind_pos   -0.016970
dtype: float64